In [2]:
#import pipline function from transformers
from transformers import pipeline

ModuleNotFoundError: No module named 'transformers'

Assignment 1.1: AI Text Intelligence Assistant (Hugging Face Model Integration)

A command-line assistant that performs four NLP tasks — **Sentiment Analysis**, **Machine Translation**,
**Text Generation**, and **Zero-Shot Text Classification** — using pre-trained Hugging Face Transformer
models.

**What this notebook demonstrates:**
- `pipeline()` API for rapid inference (all four tasks)
- `AutoTokenizer` + `AutoModelForX` manual inference, without `pipeline()` (Sentiment, Translation, Classification)
- A side-by-side comparison of `pipeline()` vs. manual `AutoModel` inference for the **same** model (Sentiment)
- A side-by-side comparison of **two different models** solving the **same** task (Classification:
  `facebook/bart-large-mnli` vs. `valhalla/distilbart-mnli-12-3`)
- Confidence scores and inference time reported for every task
- Graceful handling of empty/invalid input
- An interactive CLI loop for dynamic task selection

In [3]:
import time
import torch
from transformers import (pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,AutoModelForQuestionAnswering)

ModuleNotFoundError: No module named 'torch'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
"""
Shared model-loading helpers.

Both are simple in-memory caches keyed by task/model name so that repeated calls
during the CLI session (or repeated cells in this notebook) don't reload the same
model from the Hugging Face Hub / disk more than once."""

_pipelines = {}
_manual_models = {}  # key -> (tokenizer, model)


def get_pipeline(task, model_name):
  """
    Return a cached Hugging Face pipeline() object for (task, model_name),
    creating and caching it on first use.

    Args:
        task (str): pipeline task name, e.g. "sentiment-analysis", "text-generation".
        model_name (str): Hugging Face Hub model id, e.g. "facebook/bart-large-mnli".

    Returns:
        transformers.Pipeline: a ready-to-call pipeline instance.
    """
    key = f"{task}::{model_name}"
    if key not in _pipelines:
        print(f"[Loading pipeline] {task} -> {model_name}")
        _pipelines[key] = pipeline(task, model=model_name)
    return _pipelines[key]


def get_manual_model(model_name, model_class):
  """
    Return a cached (tokenizer, model) pair loaded via AutoTokenizer + the given
    AutoModelForX class, for manual (non-pipeline) inference.

    Args:
        model_name (str): Hugging Face Hub model id.
        model_class (type): an AutoModelForX class, e.g. AutoModelForSequenceClassification.

    Returns:
        tuple[PreTrainedTokenizer, PreTrainedModel]: the loaded tokenizer and model.
    """
    if model_name not in _manual_models:
        print(f"[Loading AutoModel] {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        _manual_models[model_name] = (tokenizer, model)
    return _manual_models[model_name]

**Sentiment Analysis**

In [1]:
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"


def run_sentiment(text):
   """
    Run sentiment analysis on `text` two ways and return a formatted comparison:
      1. pipeline() — the fast, high-level Hugging Face API.
      2. AutoTokenizer + AutoModelForSequenceClassification — manual inference,
         reproducing what the pipeline does internally (tokenize -> forward pass ->
         softmax -> argmax) to satisfy the "AutoModel without pipeline" requirement
         and to sanity-check that both paths agree.

    Args:
        text (str): input text to analyze. Empty/whitespace-only input is rejected
            gracefully with a warning message instead of raising.

    Returns:
        str: a markdown-formatted report with label, confidence (%), and inference
            time (s) for both methods.
    """
    if not text or not text.strip():
        return "⚠️ Please enter some text to analyze."

    # Method 1: pipeline()
    clf = get_pipeline("sentiment-analysis", SENTIMENT_MODEL)
    start = time.time()
    pipe_result = clf(text)[0]
    pipe_time = time.time() - start

    # Method 2: AutoTokenizer + AutoModel (manual, no pipeline)
    tokenizer, model = get_manual_model(SENTIMENT_MODEL, AutoModelForSequenceClassification)
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    start = time.time()
    with torch.no_grad():
        outputs = model(**inputs)
    manual_time = time.time() - start

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()
    manual_label = model.config.id2label[pred_id]
    manual_confidence = round(probs[pred_id].item() * 100, 2)

    return (
        f"### 🙂 Sentiment Analysis\n\n"
        f"**Method 1 — pipeline()**\n"
        f"- Model: `{SENTIMENT_MODEL}`\n"
        f"- Label: **{pipe_result['label']}**\n"
        f"- Confidence: **{round(pipe_result['score'] * 100, 2)}%**\n"
        f"- Inference time: {pipe_time:.3f}s\n\n"
        f"**Method 2 — AutoTokenizer + AutoModel (manual, no pipeline)**\n"
        f"- Model: `{SENTIMENT_MODEL}`\n"
        f"- Label: **{manual_label}**\n"
        f"- Confidence: **{manual_confidence}%**\n"
        f"- Inference time: {manual_time:.3f}s\n"
    )


# Quick test
print(run_sentiment("I absolutely loved this product, it exceeded all my expectations!"))

IndentationError: unexpected indent (1877038925.py, line 21)

**Machine Translation**

In [ ]:
TRANSLATION_MODEL = "facebook/mbart-large-50-many-to-many-mmt"
LANG_MAP = {"Hindi": "hi_IN", "French": "fr_XX", "German": "de_DE", "Spanish": "es_XX"}


def get_generation_confidence(output):
  """
    Estimate an overall confidence score (%) for a beam-search generation output.

    For each decoding step, takes the max softmax probability across the vocabulary
    (averaged across beams if more than one), then averages that value across all
    generated steps. This is a simple proxy for "how sure was the model, on average,
    about each token it generated" — not a formal sequence probability.

    Args:
        output: a `GenerateBeamOutput`/`GenerateOutput` returned by `model.generate()`
            with `return_dict_in_generate=True, output_scores=True`.

    Returns:
        float | None: confidence as a percentage rounded to 2 decimals, or None if
            no per-step scores were returned.
    """
    if not output.scores:
        return None
    probs = []
    for step_logits in output.scores:
        step_probs = torch.softmax(step_logits, dim=-1)
        # Average across beams instead of failing on multi-element tensor
        step_conf = torch.max(step_probs, dim=-1).values.mean().item()
        probs.append(step_conf)
    return round((sum(probs) / len(probs)) * 100, 2)



def run_translation(text, target_lang_name):
   """
    Translate English `text` into `target_lang_name` using mBART-50 via
    AutoTokenizer + AutoModelForSeq2SeqLM (manual inference, no pipeline).

    Args:
        text (str): English source text. Empty/whitespace-only input is rejected
            gracefully.
        target_lang_name (str): one of the keys in LANG_MAP (e.g. "Hindi", "French",
            "German", "Spanish"). Anything else is rejected gracefully.

    Returns:
        str: a markdown-formatted report with the translated text, an approximate
            confidence score, and inference time (s).
    """
    if not text or not text.strip():
        return "⚠️ Please enter text to translate."
    if target_lang_name not in LANG_MAP:
        return "⚠️ Please select a valid target language."

    target_lang_code = LANG_MAP[target_lang_name]
    tokenizer, model = get_manual_model(TRANSLATION_MODEL, AutoModelForSeq2SeqLM)
    tokenizer.src_lang = "en_XX"
    inputs = tokenizer(text, return_tensors="pt")

    start = time.time()
    output = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[target_lang_code],
        num_beams=4,
        return_dict_in_generate=True,
        output_scores=True,
    )
    elapsed = time.time() - start

    result_text = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    confidence = get_generation_confidence(output)

    return (
        f"### 🌐 Machine Translation (English → {target_lang_name})\n\n"
        f"- Model: `{TRANSLATION_MODEL}` (AutoTokenizer + AutoModel, no pipeline)\n"
        f"- Inference time: {elapsed:.3f}s\n"
        f"- Confidence: {confidence}%\n\n"
        f"**Translation:**\n{result_text}"
    )


# Quick test
print(run_translation("The invoice has been processed and sent for approval.", "Hindi"))

### 🌐 Machine Translation (English → Hindi)

- Model: `facebook/mbart-large-50-many-to-many-mmt` (AutoTokenizer + AutoModel, no pipeline)
- Inference time: 14.316s
- Confidence: 73.23%

**Translation:**
बीजक प्रक्रमित किया गया है और अनुमोदन के लिए भेजा गया है।


**Text Generation**

In [ ]:
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"


def run_text_generation(prompt):
  """
    Generate a continuation of `prompt` using Qwen2.5-0.5B-Instruct via pipeline().

    Uses top-k/top-p (nucleus) sampling with temperature so repeated calls on the
    same prompt can produce varied completions.

    Args:
        prompt (str): starting text for the model to continue. Empty/whitespace-only
            input is rejected gracefully.

    Returns:
        str: a markdown-formatted report with the generated text, sampling
            parameters used, and inference time (s).
    """
    if not prompt or not prompt.strip():
        return "⚠️ Please enter a prompt."

    generator = get_pipeline("text-generation", GENERATION_MODEL)

    start = time.time()
    result = generator(
        prompt,
        max_new_tokens=60,
        num_return_sequences=1,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=generator.tokenizer.eos_token_id,
    )[0]
    elapsed = time.time() - start

    return (
        f"### ✍️ Text Generation\n\n"
        f"- Model: `{GENERATION_MODEL}`\n"
        f"- Inference time: {elapsed:.3f}s\n"
        f"- Sampling: top_k=50, top_p=0.95, temperature=0.8\n\n"
        f"**Generated text:**\n{result['generated_text']}"
    )


# Quick test
print(run_text_generation("The future of artificial intelligence in finance is"))

[Loading pipeline] text-generation -> Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### ✍️ Text Generation

- Model: `Qwen/Qwen2.5-0.5B-Instruct`
- Inference time: 12.000s
- Sampling: top_k=50, top_p=0.95, temperature=0.8

**Generated text:**
The future of artificial intelligence in finance is bright, but it’s also a complex and evolving field that presents challenges for financial institutions. One challenge is the need to ensure that AI is used ethically and fairly across different demographics and income levels.

AI can be used to detect fraud and protect against fraudulent activities, but there are concerns about its impact


# **Comparison two different models for same task**

In [ ]:
CLASSIFICATION_MODEL_A = "facebook/bart-large-mnli"
CLASSIFICATION_MODEL_B = "valhalla/distilbart-mnli-12-3"
DEFAULT_LABELS = ["finance", "sports", "technology", "politics", "entertainment"]


def compare_classification_models(text, labels_csv):
   """
    Zero-shot classify `text` against `labels_csv` using two DIFFERENT Transformer
    models and report both results side by side.

    This satisfies the "compare outputs of different Transformer models for a
    similar task" requirement (as distinct from comparing pipeline() vs. AutoModel
    for the SAME model, which is demonstrated separately for Sentiment Analysis):
      - Model A: facebook/bart-large-mnli        (~400M params, full-size)
      - Model B: valhalla/distilbart-mnli-12-3    (~200M params, distilled)

    Both are used via pipeline("zero-shot-classification", ...), so the comparison
    isolates the effect of model choice (size/architecture) rather than API path.

    Args:
        text (str): input text to classify. Empty/whitespace-only input is rejected
            gracefully.
        labels_csv (str): comma-separated candidate labels, e.g.
            "finance, sports, technology". Falls back to DEFAULT_LABELS if blank.
            Fewer than 2 usable labels is rejected gracefully.

    Returns:
        str: a markdown-formatted report with each model's top label, confidence
            (%), inference time (s), and whether the two models agree.
    """
    if not text or not text.strip():
        return "Please enter some text to classify."

    candidate_labels = [l.strip() for l in labels_csv.split(",") if l.strip()] or DEFAULT_LABELS
    if len(candidate_labels) < 2:
        return "Please provide at least two comma-separated candidate labels."

    # Model A: facebook/bart-large-mnli
    clf_a = get_pipeline("zero-shot-classification", CLASSIFICATION_MODEL_A)
    start = time.time()
    result_a = clf_a(text, candidate_labels)
    time_a = time.time() - start
    label_a = result_a["labels"][0]
    conf_a = round(result_a["scores"][0] * 100, 2)

    # Model B: valhalla/distilbart-mnli-12-3 (distilled, lighter)
    clf_b = get_pipeline("zero-shot-classification", CLASSIFICATION_MODEL_B)
    start = time.time()
    result_b = clf_b(text, candidate_labels)
    time_b = time.time() - start
    label_b = result_b["labels"][0]
    conf_b = round(result_b["scores"][0] * 100, 2)

    agreement = "Models agree" if label_a == label_b else "Models disagree"

    return (
        f"### Text Classification — Model Comparison\n\n"
        f"**Model A — `{CLASSIFICATION_MODEL_A}`**\n"
        f"- Top label: **{label_a}** ({conf_a}%)\n"
        f"- Inference time: {time_a:.3f}s\n\n"
        f"**Model B — `{CLASSIFICATION_MODEL_B}`**\n"
        f"- Top label: **{label_b}** ({conf_b}%)\n"
        f"- Inference time: {time_b:.3f}s\n\n"
        f"**{agreement}**\n"
    )


# Quick test
print(compare_classification_models(
    "The quarterly earnings report showed a significant rise in revenue.",
    "finance, sports, technology, politics, entertainment",
))

### Text Classification — Model Comparison

**Model A — `facebook/bart-large-mnli`**
- Top label: **finance** (74.15%)
- Inference time: 6.973s

**Model B — `valhalla/distilbart-mnli-12-3`**
- Top label: **finance** (51.15%)
- Inference time: 1.643s

**Models agree**



In [ ]:
while True:
    task = input(
        "\nWhich task do you want? "
        "(sentiment / translate / generate / classify / quit): "
    ).strip().lower()

    if task == "quit":
        print("Goodbye!")
        break

    elif task == "sentiment":
        text = input("Enter text to analyze sentiment: ")
        print(run_sentiment(text))

    elif task == "translate":
        text = input("Enter English text to translate: ")
        lang = input("Enter target language (Hindi/French/German/Spanish): ")
        print(run_translation(text, lang))

    elif task == "generate":
        text = input("Enter your starting text: ")
        print(run_text_generation(text))

    elif task == "classify":
        text = input("Enter text to classify: ")
        labels = input("Enter candidate labels (comma-separated, or leave blank for default): ")
        print(compare_classification_models(text, labels))

    else:
        print("Please type one of: sentiment, translate, generate, classify, quit")

